# 05 — Proposed method: a small LoRA judge trained to be register-invariant

**Goal.** Train a small judge (about 3B parameters, QLoRA, free T4) that rates *content correctness* and gives the **same score to the same content whatever the register**.

## Training data (no overlap with the benchmark)
1. **Passage-disjoint.** Training questions come from EDRAC passages that are *not* used by the 36 benchmark questions (test + reserve), and train/validation are split by passage. No leakage.
2. **Hard negatives for free.** For each question, the *correct* answer is its gold answer (label **5**). The *incorrect* answer is the gold answer to a **different question of the same passage** (label **1**): same topic and style, but wrong for this question. No LLM is needed to create wrong answers.
3. **Register groups.** Each answer is rewritten into MSA, Franco and code-switching by a **local** open model (Qwen3-8B, greedy decoding, no API) and kept only if it passes the rule checks. One *group* = the same content in the 4 registers, with one label.
4. The reference is the gold answer in a random register (Egyptian or MSA), mirroring the two reference registers used at test time.

## Method
The judge reads the same prompt as the baselines and is trained on the 5-way digit distribution (the same read-out used at inference):

`loss = CE(label digit) + lambda * mean_over_groups Var_over_registers( expected score )`

The second term is a **register-consistency penalty**: if the same content is scored differently across registers, the model is penalised.

## Variants (this is also the ablation)
| Variant | Registers seen in training | lambda |
|---|---|---|
| `lora-egy-only` | Egyptian only | 0 |
| `lora-aug` | all 4 | 0 |
| `lora-aug-cons` | all 4 | 1.0 |

Every variant uses the same seed, the same number of optimiser steps and the same number of sequences per step, so differences come from the method and not from compute. The un-tuned base model (`zeroshot-*`) is the reference point.

**Caution on lambda.** A consistency penalty has a trivial solution: output the same score for everything. We therefore monitor the validation *separation* (mean score of correct minus incorrect) and warn if it collapses. If it does, lower lambda.

**Limitations of this first version (Milestone 1).** Training labels are only {1, 5}, so the judge has not been trained on `partial` answers; the benchmark does contain them, and they are used only at evaluation. The training registers are machine-generated, so they are cleaner than real Franco.

In [ ]:
import os, sys
REPO_DIR = "register_bias_judge"
if not os.path.exists("src/common.py"):
    if os.path.isdir(REPO_DIR):                                   # unzipped next to the notebook
        os.chdir(REPO_DIR)
    else:
        try:                                                      # Google Colab: the repo lives in your Drive, so
            from google.colab import drive                        # data/ and results/ survive between sessions
            drive.mount("/content/drive")
            os.chdir(f"/content/drive/MyDrive/{REPO_DIR}")
        except Exception as e:
            print("Could not locate the repo folder:", e)
sys.path.insert(0, os.getcwd())
assert os.path.exists("src/common.py"), "Folder 'register_bias_judge' not found. Put it in the root of your Google Drive (My Drive) and re-run."
print("working in:", os.getcwd())

In [ ]:
%pip install -q "transformers>=4.51" peft accelerate bitsandbytes datasets scipy

In [ ]:
import json
import numpy as np, pandas as pd, torch
from src import common as C
from src import models as M
from src import lora as L
from src import edrac as E

C.set_seed(C.SEED)
FOUR_BIT = torch.cuda.is_available()
GENERATOR = "Qwen/Qwen3-8B"                 # local model that writes the register variants of the training answers
BASE = "Qwen/Qwen2.5-3B-Instruct"           # the judge that gets LoRA-tuned (<= 3B fits QLoRA on a T4)
STEPS, SEQS_PER_STEP, LR = 120, 8, 2e-4
VARIANTS = {
    "lora-egy-only": dict(regs=["egyptian"], lam=0.0),
    "lora-aug":      dict(regs=C.REGISTERS,  lam=0.0),
    "lora-aug-cons": dict(regs=C.REGISTERS,  lam=1.0),
}
CKPT = C.ROOT / "checkpoints"; CKPT.mkdir(exist_ok=True)
M.hf_login()

## 1) Select training / validation questions (passage-disjoint from the benchmark)

In [ ]:
qa = E.load_edrac_qa()
pool = E.filter_qa(qa)
bench_sources = set(pd.read_csv(C.DATA / "candidates_long.csv").source_id)      # test AND reserve passages
sel = L.select_train_questions(pool, bench_sources, max_per_passage=3, val_frac=0.2, seed=C.SEED)
assert set(sel.source_id).isdisjoint(bench_sources), "leakage: a benchmark passage is in the training pool"
sel.to_csv(C.DATA / "lora_train_questions.csv", index=False, encoding="utf-8-sig")
print(sel.groupby("split").agg(questions=("question", "size"), passages=("source_id", "nunique")))
sel.head(3)[["question", "answer", "neg_answer"]]

## 2) Write the register variants of every training answer (local model, no API)

Results are cached in `data/lora_conversions.json`, so re-running is free. **Read the examples printed at the end** — machine-written Franco is the weakest point of this step.

In [ ]:
CONV_PATH = C.DATA / "lora_conversions.json"
converted = json.loads(CONV_PATH.read_text(encoding="utf-8")) if CONV_PATH.exists() else {}
need = {}
for r in sel.itertuples():
    need[r.answer] = r.question
    need[r.neg_answer] = r.neg_question
todo = [a for a in need if a not in converted]
print(f"{len(need)} distinct answers | {len(todo)} still to convert")

if todo:
    gen, gtok = M.load_causal_lm(GENERATOR, four_bit=FOUR_BIT)
    outs = L.convert_answers_local(gen, gtok, [(need[a], a) for a in todo], batch_size=8)
    for a, o in zip(todo, outs):
        converted[a] = L.validate_conversion(a, o)          # None if parsing or rule checks failed
    CONV_PATH.write_text(json.dumps(converted, ensure_ascii=False, indent=1), encoding="utf-8")
    del gen
    M.free_gpu()

ok = {a: v for a, v in converted.items() if v is not None}
print(f"{len(ok)}/{len(converted)} answers converted and passed the rule checks")
for a in list(ok)[:3]:
    print("\nEGY :", ok[a]["egyptian"]); print("MSA :", ok[a]["msa"]); print("FRA :", ok[a]["franco"]); print("CSW :", ok[a]["codeswitch"])

## 3) Build the groups

In [ ]:
groups = L.build_groups(sel, ok, seed=C.SEED)
train_groups = [g for g in groups if g["split"] == "train"]
val_groups = [g for g in groups if g["split"] == "val"]
print(f"train groups: {len(train_groups)} | val groups: {len(val_groups)} | labels: {sorted({g['label'] for g in groups})}")
assert len(train_groups) >= 20, "too few training groups: relax filters or check the conversion step"

## 4) Train the three variants and evaluate on the benchmark

For each variant: load the base model fresh (4-bit), train, check the validation diagnostics, score the 720 benchmark pairs with the **plain** prompt (no hint about registers), save the adapter. Expect roughly 5-10 minutes per variant on a T4.

In [ ]:
items = C.load_items()
pairs = C.build_pairs(items)
all_scores, diag = [], []

def score_test(model, tok, name, family):
    s, _ = M.judge_pairs(model, tok, pairs, "plain", batch_size=8, desc=name)
    return pd.DataFrame({"pair_id": pairs.pair_id, "evaluator": name, "family": family, "score": s})

def val_diag(model, tok, name):
    v = L.evaluate_groups(model, tok, val_groups)
    sep = v[v.label == 5].score.mean() - v[v.label == 1].score.mean()
    reg_std = v.groupby("group").score.std().mean()
    if sep < 1.0:
        print(f"WARNING {name}: validation separation {sep:.2f} < 1.0 -> the judge may have collapsed to a constant score (lower lambda)")
    return {"variant": name, "val_separation": sep, "val_register_std": reg_std}

# reference point: the same base model, no training
model, tok = M.load_causal_lm(BASE, four_bit=FOUR_BIT)
name = "zeroshot-" + BASE.split("/")[-1].lower()
all_scores.append(score_test(model, tok, name, "llm"))
diag.append(val_diag(model, tok, name))
del model; M.free_gpu()
pd.DataFrame(diag)

In [ ]:
for name, cfg in VARIANTS.items():
    model, tok = M.load_causal_lm(BASE, four_bit=FOUR_BIT)
    gps = max(1, SEQS_PER_STEP // len(cfg["regs"]))          # keep sequences per step equal across variants
    print(f"\n=== {name} | registers={cfg['regs']} | lambda={cfg['lam']} | groups/step={gps}")
    model, log = L.train_lora_judge(model, tok, train_groups, cfg["regs"], steps=STEPS, groups_per_step=gps,
                                    lr=LR, lam=cfg["lam"], seed=C.SEED)
    diag.append(val_diag(model, tok, name))
    all_scores.append(score_test(model, tok, name, "lora"))
    model.save_pretrained(CKPT / name)
    log.to_csv(C.RESULTS / f"train_log_{name}.csv", index=False)
    del model; M.free_gpu()

print("\nsaved:", C.save_scores(pd.concat(all_scores, ignore_index=True), "lora"))
dg = pd.DataFrame(diag); dg.to_csv(C.RESULTS / "lora_validation.csv", index=False)
dg.round(3)

## 5) Next step

Run **notebook 04** again: it now includes the LoRA judges and the zero-shot base model, and answers the research question directly — lower register penalty / inconsistency / flip rate **without** a drop in `spearman_all`.